# Music2Vec Embedding Extractor

In [1]:
!pip install muq


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import hashlib
import tempfile
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from muq import MuQMuLan
from tqdm.notebook import tqdm
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# # Google Colab
# RAW_AUDIO_DIR = '/content/drive/MyDrive/music2vec/songs'
# CACHE_DIR = '/content/drive/MyDrive/music2vec/embeddings_cache'

# On-device
RAW_AUDIO_DIR = 'songs'
CACHE_DIR = 'embeddings_cache'

os.makedirs(CACHE_DIR, exist_ok=True)

Using device: cuda


In [2]:
print("Loading MuLan model...")
mulan = MuQMuLan.from_pretrained("OpenMuQ/MuQ-MuLan-large")
mulan = mulan.to(device).eval()
model_lock = threading.Lock()
print("Model loaded.")

Loading MuLan model...


c:\Users\matth\AppData\Local\Python\pythoncore-3.10-64\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [3]:
raw_files = []
if os.path.exists(RAW_AUDIO_DIR):
    raw_files = [f for f in os.listdir(RAW_AUDIO_DIR) if f.endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a'))]

print(f"Found {len(raw_files)} audio files.")

def preprocess_audio(filename):
    in_path = os.path.join(RAW_AUDIO_DIR, filename)
    cache_path = os.path.join(CACHE_DIR, f"{filename}.pt")
    
    # Skip if already extracted/cached
    if os.path.exists(cache_path):
        return filename, cache_path, None
        
    try:
        # Load audio directly into a PyTorch tensor 
        waveform, sample_rate = torchaudio.load(in_path)
        
        # Convert to mono by averaging channels if stereo/multi-channel
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Resample to 24000 Hz if necessary
        target_sr = 24000
        if sample_rate != target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sr)
            waveform = resampler(waveform)
            
        # Normalize amplitude (Max absolute value = 1.0)
        max_val = torch.max(torch.abs(waveform))
        if max_val > 0:
            waveform = waveform / max_val
            
        wav = waveform.squeeze(0).to(dtype=torch.float32)
        return filename, cache_path, wav
    except Exception as e:
        print(f"Error handling {filename}: {e}")
        return filename, cache_path, None

if raw_files:
    max_workers = os.cpu_count() or 4
    BATCH_SIZE = 4  # Adjust per VRAM capacity limit
    
    # Decouple I/O workload and GPU inference
    batch_tensors = []
    batch_metadata = []
    
    # Multi-threading for the CPU-bound I/O/Ffmpeg tasks 
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(preprocess_audio, f) for f in raw_files]
        
        for future in tqdm(as_completed(futures), total=len(raw_files), desc="Processing Pipeline"):
            filename, cache_path, wav_tensor = future.result()
            
            if wav_tensor is not None:
                batch_tensors.append(wav_tensor)
                batch_metadata.append(cache_path)
            
            # Flush batch inference when capacity is hit
            if len(batch_tensors) >= BATCH_SIZE:
                padded_batch = torch.nn.utils.rnn.pad_sequence(batch_tensors, batch_first=True).to(device)
                
                with torch.no_grad():
                    embeds = mulan(wavs=padded_batch)
                    
                for idx, c_path in enumerate(batch_metadata):
                    # Save individually matching prior unsqueeze(0) format
                    torch.save(embeds[idx].unsqueeze(0).cpu(), c_path) 
                    
                batch_tensors.clear()
                batch_metadata.clear()
                
        # Drain any last few samples left under BATCH_SIZE
        if len(batch_tensors) > 0:
            padded_batch = torch.nn.utils.rnn.pad_sequence(batch_tensors, batch_first=True).to(device)
            with torch.no_grad():
                embeds = mulan(wavs=padded_batch)
            for idx, c_path in enumerate(batch_metadata):
                torch.save(embeds[idx].unsqueeze(0).cpu(), c_path)

print("Done!")

Found 636 audio files.


Processing Pipeline:   0%|          | 0/636 [00:00<?, ?it/s]

Done!
